In [3]:
import re

In [4]:
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common." #definindo um text

In [5]:
pattern = r"Paul [A-Z]\w+" #regra para que contenha um nome com letra maiuscula depois do Paul

In [6]:
matches = re.finditer(pattern, text) # é retornado os textos que estao dentro do padrao definido dentro de pattern
for match in matches:
    print (match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>


In [7]:
import spacy
from spacy.tokens import Span

In [8]:
nlp = spacy.blank("en")  # cria uma pipeline vazia
doc = nlp(text)  # processa o texto
print(doc.ents)  # mostra as entidades encontradas

original_ents = list(doc.ents)  # guarda as entidades originais
mwt_ents = []  # cria lista para novas entidades

for match in re.finditer(pattern, doc.text):  # procura correspondências no texto
    start, end = match.span()  # pega as posições inicial e final
    span = doc.char_span(start, end)  # converte para um Span do spaCy

    if span is not None:  # verifica se a conversão deu certo
        mwt_ents.append((span.start, span.end, span.text))  # adiciona a entidade

for ent in mwt_ents:  # percorre as novas entidades
    start, end, name = ent  # separa início, fim e texto
    per_ent = Span(doc, start, end, label="PERSON")  # define a entidade como pessoa
    original_ents.append(per_ent)  # adiciona à lista

doc.ents = original_ents  # atualiza as entidades do documento

for ent in doc.ents:  # percorre as entidades
    print(ent.text, ent.label_)  # mostra texto e classificação

()
Paul Newman PERSON
Paul Hollywood PERSON


In [9]:
print (mwt_ents)

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]


In [10]:
from spacy.language import Language

@Language.component("paul_ner")
# delara um componente a partir de uma função
def paul_ner(doc):
    pattern = r"Paul [A-Z]\w+"
    # mesma regra de antes, nome que começa com letra maiuscula
    original_ents = list(doc.ents)
    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        # pega os caracteres de inicio e fim
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="PERSON")
        original_ents.append(per_ent)
    doc.ents = original_ents
    return (doc)

In [11]:
nlp2 = spacy.blank("en") # cria uma nova pipeline
nlp2.add_pipe("paul_ner") # é adicionado a função dentro da pipeline

<function __main__.paul_ner(doc)>

In [12]:
doc2 = nlp2(text) # cria outro doc
print (doc2.ents)

(Paul Newman, Paul Hollywood)


In [13]:
from spacy.language import Language  # importa a classe Language
from spacy.util import filter_spans  # filtra entidades sobrepostas

@Language.component("cinema_ner")  # registra o componente no spaCy
def cinema_ner(doc):  # cria a função do componente
    pattern = r"Hollywood"  # procura pela palavra Hollywood

    original_ents = list(doc.ents)  # guarda as entidades existentes
    mwt_ents = []  # cria lista para novas entidades

    for match in re.finditer(pattern, doc.text):  # procura o padrão no texto
        start, end = match.span()  # pega início e fim da correspondência
        span = doc.char_span(start, end)  # converte para Span do spaCy

        if span is not None:  # verifica se a conversão deu certo
            mwt_ents.append((span.start, span.end, span.text))  # adiciona a entidade

    for ent in mwt_ents:  # percorre as novas entidades
        start, end, name = ent  # separa início, fim e texto
        per_ent = Span(doc, start, end, label="CINEMA")  # define como CINEMA
        original_ents.append(per_ent)  # adiciona à lista

    filtered = filter_spans(original_ents)  # remove entidades sobrepostas
    doc.ents = filtered  # atualiza as entidades do documento

    return doc  # retorna o documento modificado

In [14]:
nlp3 = spacy.load("en_core_web_sm") # carrega o modelo en_core_web
nlp3.add_pipe("cinema_ner") # adiciona a função dentro da pipeline

<function __main__.cinema_ner(doc)>

In [16]:
doc3 = nlp3(text) # cria outro doc
for ent in doc3.ents:
    print (ent.text, ent.label_) # mostra o texto e o rótulo

Paul Newman PERSON
American NORP
Paul Hollywood PERSON
British NORP
Paul PERSON
